# 🍲 BeautifulSoup — Web Scraping
## Python Ecosystem Tutorial Series — Module 8 of 18

**Author:** Himanshu Goel | [himanshugoel.github.io](https://himanshugoel.github.io)

---

| | |
|---|---|
| **Library** | 🍲 BeautifulSoup |
| **Domain** | Web Scraping |
| **Dataset** | Drug safety table |
| **Module** | 8 of 18 |

**What you will learn:**

1. What BeautifulSoup is and why it exists
2. Core concepts and data structures
3. Hands-on code with real data
4. Visualisations and interpretation
5. When to use it and alternatives

```bash
# Install required libraries
pip install beautifulsoup4
```

## Quick Reference Card

| Code | What it does |
|------|--------------|
| `BeautifulSoup(html)` | Parse HTML |
| `soup.find(tag)` | First match |
| `soup.find_all(tag)` | All matches |
| `soup.select(css)` | CSS selector |
| `element.text` | Get text content |

# 8. 🍲 BeautifulSoup — Web Scraping
> **Python + BeautifulSoup = Web Scraping**

BeautifulSoup parses HTML/XML. Combined with `requests`, it lets you extract
data from any website — prices, articles, research papers, tables.

**Key concepts:** HTML parsing, CSS selectors, find/find_all, navigate the DOM tree

In [ ]:
from bs4 import BeautifulSoup
import requests
import re
import matplotlib.pyplot as plt
import pandas as pd

# ── Understanding HTML structure first ────────────────────────────────────────
sample_html = """
<html>
<body>
  <h1>Drug Safety Database</h1>
  <table id="drug-table">
    <tr>
      <th>Drug</th><th>Category</th><th>LD50 (mg/kg)</th><th>Status</th>
    </tr>
    <tr class="safe">
      <td>Aspirin</td><td>NSAID</td><td>200</td><td>Approved</td>
    </tr>
    <tr class="warning">
      <td>Acetaminophen</td><td>Analgesic</td><td>338</td><td>Approved</td>
    </tr>
    <tr class="danger">
      <td>Warfarin</td><td>Anticoagulant</td><td>3</td><td>Approved</td>
    </tr>
    <tr class="safe">
      <td>Caffeine</td><td>Stimulant</td><td>192</td><td>GRAS</td>
    </tr>
    <tr class="danger">
      <td>Atrazine</td><td>Herbicide</td><td>1869</td><td>Regulated</td>
    </tr>
  </table>
  <p class="note">Data from ToxValDB | Last updated: 2024</p>
</body>
</html>
"""

# ── Parse with BeautifulSoup ──────────────────────────────────────────────────
soup = BeautifulSoup(sample_html, "html.parser")

print("── Find by tag ──")
print(soup.find("h1").text)

print("\n── Find all table rows ──")
rows = soup.find_all("tr")
print(f"Found {len(rows)} rows")

print("\n── Extract data from table ──")
records = []
for row in rows[1:]:   # skip header row
    cols = row.find_all("td")
    if cols:
        records.append({
            "drug":     cols[0].text,
            "category": cols[1].text,
            "ld50":     int(cols[2].text),
            "status":   cols[3].text,
            "class":    row.get("class", ["unknown"])[0]
        })
        print(f"  {records[-1]}")

df_scraped = pd.DataFrame(records)
print(f"\n── Scraped DataFrame ──")
print(df_scraped)

In [ ]:
# ── Real web scraping: fetch from Wikipedia ──────────────────────────────────
print("── Scraping Wikipedia (Nobel Prize in Chemistry) ──")

try:
    url = "https://en.wikipedia.org/wiki/Nobel_Prize_in_Chemistry"
    headers = {"User-Agent": "Mozilla/5.0 (Educational scraping demo)"}
    resp = requests.get(url, headers=headers, timeout=10)
    soup_wiki = BeautifulSoup(resp.content, "html.parser")

    # Find the first data table
    table = soup_wiki.find("table", {"class": "wikitable"})
    if table:
        rows = table.find_all("tr")[:15]  # first 15 rows
        print("Recent Nobel laureates in Chemistry:")
        for row in rows[1:8]:   # skip header
            cells_r = row.find_all(["td","th"])
            if cells_r:
                year_cell = cells_r[0].get_text(strip=True)[:4]
                # Name is usually in 3rd column
                if len(cells_r) >= 3:
                    name = cells_r[2].get_text(strip=True)[:40]
                    print(f"  {year_cell}: {name}")
except Exception as e:
    print(f"Network not available ({e}) — showing pattern instead")
    # Pattern you would use:
    example_data = [
        (2024, "David Baker, Demis Hassabis, John Jumper"),
        (2023, "Moungi Bawendi, Louis Brus, Alexei Ekimov"),
        (2022, "Carolyn Bertozzi, Morten Meldal, Barry Sharpless"),
    ]
    for year, names in example_data:
        print(f"  {year}: {names}")

# ── Visualise scraped drug data ───────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

class_colors = {"safe":"#27AE60","warning":"#F1C40F","danger":"#E74C3C"}
cols_bar = [class_colors.get(c,"#95A5A6") for c in df_scraped["class"]]
axes[0].barh(df_scraped["drug"], df_scraped["ld50"], color=cols_bar,
              alpha=0.85, edgecolor="white")
axes[0].set_xlabel("LD50 (mg/kg) — higher = safer")
axes[0].set_title("Scraped Drug Safety Data\n(lower LD50 = more toxic)", fontweight="bold")
axes[0].grid(True, alpha=0.3, axis="x")
for patch in [plt.Rectangle((0,0),1,1,color=v,alpha=0.85) for v in class_colors.values()]:
    pass
legend_patches = [plt.Rectangle((0,0),1,1,color=v,label=k,alpha=0.85)
                  for k,v in class_colors.items()]
axes[0].legend(handles=legend_patches, fontsize=8)

# Category pie chart
counts = df_scraped["category"].value_counts()
axes[1].pie(counts.values, labels=counts.index, autopct="%1.0f%%",
             startangle=90, textprops={"fontsize":9})
axes[1].set_title("Drug Categories in Scraped Data", fontweight="bold")

plt.suptitle("BeautifulSoup — Web Scraping Results", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("beautifulsoup_scraping.png", dpi=120, bbox_inches="tight")
plt.show()
print("\nKey pattern: requests.get(url) → BeautifulSoup(html) → find/find_all → extract")

## Deep Dive: BeautifulSoup

### HTML Parse Tree Navigation
BeautifulSoup turns raw HTML text into a navigable tree:
```python
soup.find("table")              # first matching tag
soup.find_all("tr")             # list of ALL matching tags
soup.find("div", id="main")     # match by attribute
soup.select("table.data td")    # CSS selector (most powerful)
element.text                    # get text content (strips tags)
element.get("href")             # get attribute value
element.find_next("td")         # find next sibling/descendant
```

### Parsing Libraries
```python
BeautifulSoup(html, "html.parser")   # built-in Python, slower
BeautifulSoup(html, "lxml")          # C-based, fastest, recommended
BeautifulSoup(html, "html5lib")      # most lenient, slowest
```
Install lxml: `pip install lxml`

### Static vs Dynamic Sites
| Tool | Use when |
|------|---------|
| BeautifulSoup | HTML is in the page source (Ctrl+U shows data) |
| Selenium | Data loaded by JavaScript after page loads |
| Scrapy | Crawling hundreds/thousands of pages |

### Responsible Scraping
1. Check robots.txt: `https://site.com/robots.txt`
2. Add delays: `time.sleep(1)` between requests
3. Set a User-Agent header
4. Use official APIs when available

### Real Chemical Data APIs
- PubChem: `pubchem.ncbi.nlm.nih.gov/rest/pug/compound/name/aspirin/JSON`
- ChEMBL: `www.ebi.ac.uk/chembl/api/data/molecule?pref_name=aspirin`
- EPA CompTox: `comptox.epa.gov/dashboard/`


## ✅ Key Takeaways — 🍲 BeautifulSoup

1. BeautifulSoup works for static HTML — use Selenium for JavaScript-heavy pages
2. CSS selectors are more powerful and readable than nested find() calls
3. Always check robots.txt and add rate limiting to be a responsible scraper
4. Official APIs (PubChem, ChEMBL) are always preferable to scraping

---
*Next: Continue to Module 9 of 18 in the Python Ecosystem Tutorial Series*  
*Portfolio: [himanshugoel.github.io](https://himanshugoel.github.io)*